# SWITCH ENERGY-X (INDIA): Recovering the Hidden Physics**Final result: private RMSE 0.48008 — 11th of 93 teams.**This competition posed a genuine inverse problem rather than a supervised learning task:400,000 unlabeled training rows, 100,000 test rows, 25 anonymized sensors, ~26% of cellsmissing, and **a target variable that appears in no file, in any form**.There is nothing to fit and nothing to cross-validate against.This notebook reproduces the part of the solution that needs **no labels and nosubmissions**: recovering the generator's internal physics from the feature columnsalone. Everything below runs end-to-end on the competition data.Full pipeline, methodology and post-mortem:**https://github.com/adarshcod30/switch-energy-x-india**

In [ ]:
import numpy as np, pandas as pdpd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)BASE = '/kaggle/input/switch-energy-x-india'train = pd.read_csv(f'{BASE}/train.csv')test  = pd.read_csv(f'{BASE}/test.csv')ddict = pd.read_csv(f'{BASE}/data_dictionary.csv')full  = pd.concat([train, test], ignore_index=True)F = [f'feature_{i:02d}' for i in range(1, 26)]print(f'train {train.shape}   test {test.shape}')print(f'missing fraction: {full[F].isna().mean().mean():.4f}')ddict

## 1. The data dictionary de-anonymizes everythingThis is the single most important artifact in the competition. Every column has a statedphysical role — irradiance, ambient temperature, wind speed, battery state-of-charge,demand, inverter efficiency — and four are explicitly declared **distractors**.That means the generator's structure is not a black box. It can be reverse-engineeredwith regression, using no target information at all.

In [ ]:
d = {c: full[c].values.astype(float) for c in F}def fit(y_name, X, label):    """OLS of one column on a set of candidate expressions. Reports R2 and residual sd."""    y = d[y_name]    m = np.isfinite(y)    for v in X.values(): m &= np.isfinite(v)    A = np.column_stack([np.ones(m.sum())] + [v[m] for v in X.values()])    coef, *_ = np.linalg.lstsq(A, y[m], rcond=None)    r = y[m] - A @ coef    r2 = 1 - r.var() / y[m].var()    print(f'{label:46s} n={m.sum():7d}  R2={r2:.5f}  resid_sd={r.std():.5f}')    print(f'{"":46s} coef = {np.round(coef, 6)}')    return coef, r2

## 2. Recovering the generator's identitiesEach of these is a relationship the generator used internally, recovered purely byregression across all 500,000 rows.

In [ ]:
print('--- Generation balance ---')fit('feature_14', {'solar': d['feature_12'], 'wind': d['feature_13']},    'f14 = f12 + f13  (gross = solar + wind)')print('\n--- Grid frequency tracks supply-demand imbalance ---')fit('feature_21', {'gross': d['feature_14'], 'demand': d['feature_09']},    'f21 = 50 + 0.9*(f14 - f09)')print('\n--- PV temperature derating ---')fit('feature_15', {'Tpanel': d['feature_07']},    'f15 = 1 + gamma*(T_panel - 25)')print('\n--- Faiman panel temperature (with wind cooling) ---')fit('feature_07', {'Tamb': d['feature_02'], 'G': d['feature_01'], 'v': d['feature_04']},    'f07 = f02 + a*f01 - b*f04')print('\n--- Ideal gas law ---')rho = d['feature_05'] * 100 / (287.05 * (d['feature_02'] + 273.15))fit('feature_10', {'rho': rho}, 'f10 = P/(287.05*T)')print('\n--- Declared engineered interactions ---')fit('feature_17', {'GH': d['feature_01'] * d['feature_03'] / 100}, 'f17 = f01*f03/100')fit('feature_18', {'v2': d['feature_04'] ** 2},                     'f18 = f04^2')fit('feature_19', {'ds': d['feature_08'] * d['feature_09']},        'f19 = f08*f09')

### What just came out of anonymized columns| Identity | R² | Physical meaning ||---|---|---|| `f14 = f12 + f13` | 0.99992 | Gross generation = solar + wind || `f21 = 49.99 + 0.900·f14 − 0.880·f09` | 0.99983 | Grid frequency rises with generation surplus || `f15 = 1.104 − 0.00418·f07` | 0.9766 | **PV temperature coefficient = −0.418 %/°C** || `f07 = f02 + 0.0219·f01 − 0.348·f04` | 0.9915 | Faiman cell temperature with wind cooling || `f10 = f05·100/(287.05·(f02+273.15))` | 0.9301 | **R = 287.05 J/(kg·K)** — dry air gas constant |The recovered PV coefficient of **−0.418 %/°C** sits squarely inside the published0.4–0.5 %/°C range for crystalline silicon, and the irradiance coefficient 0.0219corresponds to NOCT ≈ 37.5 °C. The simulator is physically grounded, which validatesevery downstream inference.

## 3. Where the noise livesDerived columns fit *each other* far more tightly than they fit raw sensors. Thatasymmetry is diagnostic: the generator computed derived columns from **clean latentvalues**, then added observation noise to the raw sensors separately.Practical consequence: `sqrt(f18)` is a **better** estimate of true wind speed than the`f04` sensor reading itself.

In [ ]:
def R2(y, p):    m = np.isfinite(y) & np.isfinite(p); y, p = y[m], p[m]    A = np.column_stack([np.ones(len(y)), p])    c, *_ = np.linalg.lstsq(A, y, rcond=None)    return 1 - (y - A @ c).var() / y.var()v_raw = d['feature_04']v_der = np.sqrt(np.clip(d['feature_18'], 0, None))print('Which wind-speed estimate better predicts f13 (clean wind generation)?')print(f'  f04 raw sensor      : R2 = {R2(d["feature_13"], v_raw ** 3):.6f}')print(f'  sqrt(f18) inverted  : R2 = {R2(d["feature_13"], v_der ** 3):.6f}   <-- cleaner')resid = d['feature_14'] - (d['feature_12'] + d['feature_13'])resid = resid[np.isfinite(resid)]print(f'\nsd(f14 - (f12+f13)) = {resid.std():.5f}')print(f'  => per-column sensor noise ~ {resid.std()/np.sqrt(3):.5f}, '      f'about {100*resid.std()/np.sqrt(3)/np.nanstd(d["feature_14"]):.2f}% of f14 scale')

## 4. The turbine power curve and the transmission-loss clipTwo features turn out to be **piecewise**, which matters enormously — smooth modelscannot represent them.

In [ ]:
m = np.isfinite(v_der) & np.isfinite(d['feature_13'])vv, ww = v_der[m], d['feature_13'][m]print('Wind generation vs wind speed (turbine power curve):')print(f"{'v (m/s)':>9s} {'n':>7s} {'mean f13':>10s}")for lo in range(0, 17):    s = (vv >= lo) & (vv < lo + 1)    if s.sum() > 30:        print(f'{lo:9d} {s.sum():7d} {ww[s].mean():10.3f}')print('  => cut-in ~2.0 m/s, rated ~10.75 m/s, saturates then falls (cut-out ~18)')G, XL = d['feature_01'], d['feature_22']m = np.isfinite(G) & np.isfinite(XL)print('\nTransmission loss vs irradiance:')print(f"{'G':>10s} {'n':>7s} {'mean f22':>10s} {'sd':>9s}")for lo in range(0, 1300, 150):    s = (G[m] >= lo) & (G[m] < lo + 150)    if s.sum() > 100:        print(f'{lo:10d} {s.sum():7d} {XL[m][s].mean():10.5f} {XL[m][s].std():9.5f}')print('  => rises linearly, then CLIPS HARD at 0.080 (within-bin sd collapses)')

## 5. `row_id` is not an index — it encodes timeThe dictionary labels `feature_24` a *"slow instrument calibration drift"*, which impliesthe rows have an order. Testing that order against every column reveals that **fourfeatures are exact sinusoids in the row index**, while the other 21 were drawn i.i.d.This is information present in **no feature column**, and it materially improvesimputation of the two weakest high-leverage columns.

In [ ]:
n = len(full); rid = np.arange(n, dtype=float)def phase_r2(v, period, nb=40):    m = np.isfinite(v)    b = np.minimum(((rid % period) / period * nb).astype(int), nb - 1)    df = pd.DataFrame({'v': v[m], 'b': b[m]})    return 1 - (df.v - df.groupby('b')['v'].transform('mean')).var() / df.v.var()periods = [250000, 166666, 125000, 100000, 83333, 62500, 50000, 41666, 25000, 10000, 5000]print(f"{'feature':12s} {'best phase R2':>14s} {'period (rows)':>15s}")for c in F:    r2, p = max((phase_r2(d[c], pp), pp) for pp in periods)    flag = '   <== PERIODIC' if r2 > 0.02 else ''    if r2 > 0.02 or c in ('feature_01', 'feature_13'):        print(f'{c:12s} {r2:14.4f} {p:15d}{flag}')

In [ ]:
# Shuffle control: confirm the periodicity is real, not an artifactdef autocorr(v, k):    a, b = v[k:], v[:-k]    m = np.isfinite(a) & np.isfinite(b)    return float(np.corrcoef(a[m], b[m])[0, 1])v = d['feature_08'].copy()vs = v.copy(); np.random.default_rng(0).shuffle(vs)print('Battery SOC (f08) autocorrelation:')print(f'  real     : lag1={autocorr(v,1):+.4f}  lag200={autocorr(v,200):+.4f}  lag5000={autocorr(v,5000):+.4f}')print(f'  shuffled : lag1={autocorr(vs,1):+.4f}  lag200={autocorr(vs,200):+.4f}  lag5000={autocorr(vs,5000):+.4f}')print('  => structure is real; flat (non-decaying) autocorrelation = pure sinusoid')

## 6. Measuring a target that is never shownWith no labels, the **only** observable signal about the target is the RMSE returned fora submission. That scalar inverts exactly.For a prediction vector `p` and hidden target `y`:$$\mathrm{RMSE}^2 = (\mu_y - \mu_p)^2 + \sigma_y^2 + \sigma_p^2 - 2\,\mathrm{cov}(y, p)$$Every term except `cov(y, p)` is known — so **each submission yields one exact covariancemeasurement**. Standardising `p` to the organizer-stated moments (mean 3.9, sd 4.6)reduces this to a correlation readout:$$\rho = 1 - \frac{\mathrm{RMSE}^2}{2\sigma_y^2}$$### Validating the instrument before trusting itSubmitting `feature_20` — a column the dictionary explicitly declares a **synthetic noisedistractor** — returned **ρ = −0.0025 ≈ 0**, confirming the framework is unbiased andsimultaneously pinning σ_y = 4.611 against the stated ≈4.6. A biased framework would haveshown spurious signal on a known-null feature.Repeated at the end of the search with three distractors combined: **γ = +0.0001**.

### Measured correlations with the hidden target| feature | ρ | feature | ρ | feature | ρ ||---|---|---|---|---|---|| f13 wind gen | **+0.830** | f15 temp-loss | **+0.571** | f01 irradiance | −0.286 || f21 grid freq | **+0.828** | f10 air density | +0.360 | f17 irr×hum | −0.278 || f18 wind² | **+0.806** | f03 humidity | −0.050 | f12 solar gen | −0.251 || f04 wind speed | **+0.797** | f22 trans. loss | −0.109 | f19 dem×stor | −0.174 || f07 panel temp | **−0.568** | f09 demand | −0.283 | f02 ambient T | −0.404 || **f20 (control)** | **−0.003** | f11, f08, f06 | ≈ 0 | | |Three structural findings fall out:1. **The equation is additive, not multiplicative.** The textbook merger   `f14·f15·f11·(1−f22)` reaches ρ = 0.831 — barely above raw wind alone (0.829).   Stacking efficiency factors adds nothing.2. **Solar generation is suppressed.** `f12` shows ρ = −0.251 despite a large positive   structural coefficient: irradiance raises generation but also heats the panel, and the   thermal penalty dominates the marginal correlation.3. **`f11`, `f08`, `f06` are not in the equation** (ρ ≈ 0.011, 0.006, 0.010) despite   labels implying relevance.

## 7. The dominant nonlinearity: a thermal cutoffPerturbing the best model along orthonormal directions and reading back the scoreisolated the largest single piece of unexplained variance:```relu(f07 - 55)     gamma = -0.622     explains 0.386 of target variance```Reading the *shape* of the accumulated correction shows the slope **accelerates** pastthe knee — which means the penalty is **quadratic**, not a linear hinge:| f07 range | slope of correction ||---|---|| 25–45 °C | +0.067 /°C || 45–52 °C | −0.144 /°C || 52–58 °C | −0.188 /°C || 58–70 °C | −0.319 /°C |A grid search over cell-temperature definitions found the best fit at`T_cell = T_amb + 0.020·G` with a quadratic penalty above ≈35 °C (R² = 0.8625 against themeasured correction) — physically a hard derating threshold where panels and invertersthrottle to protect themselves.

## 8. Result, and the honest limit| | Score | Rank ||---|---|---|| **Private leaderboard** (70%) | **0.48008** | **11 / 93** || Public leaderboard (30%) | 0.46441 | 10 / 93 |Trajectory: 2.757 → 1.358 → 1.070 → 0.810 → 0.719 → 0.562 → 0.508 → **0.464**### Why it plateauedOffline formula search was pushed to exhaustion and **provably cannot** identify theequation from these measurements:- Measurement noise floor on the relative moment residual: **0.00679**- Constrained symbolic search reached **0.00517–0.00576** — *below* the floor- **2,775 distinct formulas** fit equally well; two independent runs returned entirely  different equations at identical qualityFitting below the noise floor is fitting noise. Only new submissions measuring genuinelynew directions can discriminate — and each buys roughly 0.02–0.03 RMSE against a gapof ~0.47.**The structural lesson.** Teams finishing ahead used very few submissions — the winnerused 5. A handful of scalar scores cannot fit an equation, so they did not *measure* it:they **derived** the closed form from the published physics named in the data dictionaryand spent submissions calibrating a few coefficients. Deriving is dramatically moresubmission-efficient than measuring direction by direction, and that difference accountsfor the gap.---Full pipeline, methodology, recovered feature dictionary and post-mortem:### https://github.com/adarshcod30/switch-energy-x-india